In [ ]:
# ============================================
# ROLL NO: 24XXX (Replace with your roll no)
# SCENARIO 1 – MULTINOMIAL NAÏVE BAYES
# ============================================

# 1. Import Libraries
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# 2. Load Dataset
df = pd.read_csv("spam.csv", encoding="latin-1")[['v1','v2']]
df.columns = ['label', 'message']

# 3. Text Cleaning Function
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{string.punctuation}]", "", text)
    text = re.sub(r'\d+', '', text)
    return text

df['cleaned_message'] = df['message'].apply(clean_text)

# 4. Convert Text to Numerical Features (TF-IDF)
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['cleaned_message'])

# 5. Encode Target Labels
le = LabelEncoder()
y = le.fit_transform(df['label'])  # spam=1, ham=0

# 6. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. Train Multinomial Naïve Bayes
model = MultinomialNB(alpha=1.0)   # Laplace smoothing alpha=1
model.fit(X_train, y_train)

# 8. Predictions
y_pred = model.predict(X_test)

# 9. Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 10. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Multinomial NB")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# 11. Misclassified Examples
misclassified = df.iloc[X_test.indices][y_test != y_pred]
print("Some Misclassified Messages:\n", misclassified.head())

# 12. Feature Importance (Top Spam Words)
feature_names = vectorizer.get_feature_names_out()
spam_class_index = 1

topn = 15
top_spam = np.argsort(model.feature_log_prob_[spam_class_index])[-topn:]

print("Top words influencing Spam:")
for i in top_spam:
    print(feature_names[i])